# B02 · 数据结构与文件

> 阶段〇第 2 周。上周是「单个量怎么算」，本周是「一堆数据怎么组织、怎么存盘、读坏了怎么办」。
> 这些能力直接决定你后面写 RL 训练脚本时，管理超参数、日志、检查点是否顺手。

## 学习目标

1. 根据场景选择容器：`list` / `tuple` / `dict` / `set`；
2. 用推导式简洁地生成与过滤序列；
3. 用 `enumerate` / `zip` 并行遍历多个序列；
4. 用 `pathlib` 读写 txt / CSV / JSON 文件（仿真参数存 JSON、结果存 CSV）；
5. 用 `try/except` 写出「读坏了也不崩」的健壮代码；
6. 把自己写的函数整理成 `.py` 模块并 `import` 复用。

## 1. list 与 tuple：可变 vs 不可变

- `list`：**可变**序列。适合「随时间增长的数据流」——采样值、日志行、训练回报曲线。
- `tuple`：**不可变**序列。适合「一组固定的物理参数」——一旦定义就不该被改，
  不可变性本身就是保险：别人（或三个月后的你）误改它会直接报错。

In [1]:
# list：可变 —— 收集采样值
samples = [0.0, 0.18, 0.33]
samples.append(0.45)     # 尾部追加
samples[0] = 0.01        # 修改元素
print(samples, "长度:", len(samples))

# tuple：不可变 —— 一组电机参数 (转动惯量 J, 阻尼 b, 力矩常数 Kt)
motor_params = (0.01, 0.1, 0.5)
J, b, Kt = motor_params          # 解包：按位置拆开
print(f"J = {J}, b = {b}, Kt = {Kt}")
# motor_params[0] = 1.0   # 取消注释试试：TypeError，不可变对象不允许修改

[0.01, 0.18, 0.33, 0.45] 长度: 4
J = 0.01, b = 0.1, Kt = 0.5


## 2. dict 与 set

- `dict`（字典）：**键 → 值**的映射。是存「参数表 / 配置 / 实验记录」的主力，
  后面 SB3 的超参数、Isaac 的任务配置全是嵌套 dict。
- `set`（集合）：**不重复**元素的集合，常用来去重、做成员判断。

In [2]:
# dict：一阶环节的参数表
plant = {"name": "first_order", "K": 2.0, "tau": 0.5, "unit": "degC"}
print(plant["K"])                          # 按键取值（键不存在会 KeyError）
plant["Ts"] = 0.01                         # 新增键值对
print(plant.get("dead_time", "未建模"))    # get 给默认值，安全

for key, value in plant.items():           # 遍历键值对
    print(f"  {key}: {value}")

# set：传感器型号去重
ids = ["PT100", "PT100", "DS18B20", "PT100", "SHT30"]
unique_ids = set(ids)
print("去重后:", unique_ids)
print("PT100 在线?", "PT100" in unique_ids)

2.0
未建模
  name: first_order
  K: 2.0
  tau: 0.5
  unit: degC
  Ts: 0.01
去重后: {'DS18B20', 'PT100', 'SHT30'}
PT100 在线? True


## 3. 推导式（comprehension）：一行生成序列

`[表达式 for x in 序列 if 条件]` 是 Python 的招牌语法，
比「新建空列表 + for + append」更短、通常也更快。dict / set 也有对应写法。

In [3]:
xs = list(range(10))
squares = [x ** 2 for x in xs]              # 映射：每个元素算一遍
evens = [x for x in xs if x % 2 == 0]       # 过滤：只留偶数
print("平方:", squares)
print("偶数:", evens)

# dict 推导式：不同时间常数 -> 2% 调节时间（≈ 4τ）
taus = [0.1, 0.2, 0.5, 1.0]
ts_table = {f"tau={t}": round(4 * t, 2) for t in taus}
print(ts_table)

# 实际场景：从一组读数里挑出超温报警的样本
temps = [23.1, 79.8, 81.2, 25.0, 90.5]
alarms = [t for t in temps if t > 80.0]
print("报警样本:", alarms)

平方: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
偶数: [0, 2, 4, 6, 8]
{'tau=0.1': 0.4, 'tau=0.2': 0.8, 'tau=0.5': 2.0, 'tau=1.0': 4.0}
报警样本: [81.2, 90.5]


## 4. `enumerate` 与 `zip`：并行遍历

- `zip(a, b)`：把两个序列**对齐**成一对对（时间轴 ↔ 信号值）；
- `enumerate(xs)`：遍历时同时拿到**下标**和值（标注第几个采样点）。

In [4]:
t = [0.0, 0.1, 0.2, 0.3]
y = [0.0, 0.36, 0.66, 0.90]

for k, (tk, yk) in enumerate(zip(t, y)):
    print(f"样本 {k}: t = {tk:.1f} s, y = {yk:.2f}")

# zip 的经典用法：快速拼一个参数 dict
keys = ["Kp", "Ki", "Kd"]
vals = [1.0, 0.5, 0.05]
pid = dict(zip(keys, vals))
print(pid)

样本 0: t = 0.0 s, y = 0.00
样本 1: t = 0.1 s, y = 0.36
样本 2: t = 0.2 s, y = 0.66
样本 3: t = 0.3 s, y = 0.90
{'Kp': 1.0, 'Ki': 0.5, 'Kd': 0.05}


## 5. 文件读写：txt / CSV / JSON（pathlib 版）

工程惯例：

| 文件类型 | 存什么 | Python 工具 |
|---------|--------|-------------|
| `.txt` | 日志、备注 | `Path.read_text / write_text` |
| `.csv` | 表格数据（仿真结果、传感器记录） | `csv` 模块（B04 会换成 pandas） |
| `.json` | 参数配置（嵌套 dict） | `json` 模块 |

`pathlib.Path` 用 `/` 拼路径，跨平台，告别字符串拼接。本项目约定：
**一切运行产物写入 `runs/` 目录**（已 gitignore，不会污染仓库）。下面先定位项目根目录，
以后每个 notebook 都会用同样的两行。

In [5]:
from pathlib import Path
import csv
import json

# 向上找到含 pyproject.toml 的目录 = 项目根；产物统一放 runs/00_basic/
PROJECT_ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
DATA = PROJECT_ROOT / "runs" / "00_basic" / "b02"
DATA.mkdir(parents=True, exist_ok=True)
print("数据目录:", DATA)

# ---- txt：写日志、读日志 ----
log_path = DATA / "run_log.txt"
log_path.write_text("10:00:00 仿真开始\n10:00:05 仿真结束\n", encoding="utf-8")
print(log_path.read_text(encoding="utf-8"))

# ---- CSV：把一组阶跃响应数据写出去再读回来 ----
csv_path = DATA / "step_response.csv"
rows = [(round(0.1 * k, 1), round(1 - 2.718281828 ** (-0.1 * k / 0.5), 4))
        for k in range(11)]
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["t", "y"])   # 表头
    writer.writerows(rows)

with csv_path.open(encoding="utf-8") as f:
    for i, row in enumerate(csv.reader(f)):
        print(row)
        if i >= 3:
            print("...  (略)")
            break

# ---- JSON：保存 / 加载 PID 参数 ----
json_path = DATA / "pid_config.json"
json_path.write_text(json.dumps(pid, indent=2, ensure_ascii=False), encoding="utf-8")
loaded = json.loads(json_path.read_text(encoding="utf-8"))
print("读回的参数:", loaded, type(loaded))

数据目录: /data/wangf/robot_rl_learn/runs/00_basic/b02
10:00:00 仿真开始
10:00:05 仿真结束

['t', 'y']
['0.0', '0.0']
['0.1', '0.1813']
['0.2', '0.3297']
...  (略)
读回的参数: {'Kp': 1.0, 'Ki': 0.5, 'Kd': 0.05} <class 'dict'>


## 6. 异常处理：`try / except`

真实工程里文件会缺失、内容会脏、串口会掉线。Python 的风格是 **EAFP**：
「先做了再说，出错就捕获」，而不是事前层层判断。

原则：**捕获要具体**（`except FileNotFoundError` 而不是裸 `except:`），
**失败要有兜底**（返回默认值并打印警告），别把异常默默吞掉。

In [6]:
def read_gain(path, default=1.0):
    '''从文件读增益值；文件缺失或内容不是数字时返回默认值。'''
    try:
        text = Path(path).read_text(encoding="utf-8").strip()
        return float(text)
    except FileNotFoundError:
        print(f"[警告] 文件不存在: {path}，使用默认增益 {default}")
        return default
    except ValueError:
        print(f"[警告] 文件内容不是数字: {path}，使用默认增益 {default}")
        return default

(DATA / "gain.txt").write_text("2.5", encoding="utf-8")
print(read_gain(DATA / "gain.txt"))          # 正常：2.5
print(read_gain(DATA / "not_exist.txt"))     # 触发 FileNotFoundError
(DATA / "gain.txt").write_text("abc", encoding="utf-8")
print(read_gain(DATA / "gain.txt"))          # 触发 ValueError

2.5
[警告] 文件不存在: /data/wangf/robot_rl_learn/runs/00_basic/b02/not_exist.txt，使用默认增益 1.0
1.0
[警告] 文件内容不是数字: /data/wangf/robot_rl_learn/runs/00_basic/b02/gain.txt，使用默认增益 1.0
1.0


## 7. 简单模块化：`import` 自己写的 `.py`

当一段函数要在多个 notebook / 脚本里复用时，就把它写进 `.py` 文件再 `import`。
这也是本项目的工程约定：**可复用代码放 `src/`，notebook 只做实验与讲解**。

下面演示完整流程：写一个 `signal_utils.py` 模块 → 加入搜索路径 → import 使用。
（正式项目里模块放 `src/robot_rl_learn/` 并通过 editable install 引入，不需要改 `sys.path`。）

In [7]:
import sys

MOD_DIR = DATA / "modules"
MOD_DIR.mkdir(parents=True, exist_ok=True)

(MOD_DIR / "signal_utils.py").write_text('''# 小工具模块：一阶环节相关函数

import math


def step_response(t, K=1.0, tau=1.0):
    # 一阶惯性环节阶跃响应 y(t) = K(1 - e^{-t/tau})
    return K * (1.0 - math.exp(-t / tau))


def settling_time(tau, band=0.02):
    # 近似调节时间：2% 误差带取 4τ，5% 误差带取 3τ
    return (4 if band <= 0.02 else 3) * tau
''', encoding="utf-8")

# 把模块目录加入搜索路径后 import（正式项目用 src 布局 + uv 安装，无需这步）
sys.path.insert(0, str(MOD_DIR))
import signal_utils

print("y(2s, tau=0.5) =", round(signal_utils.step_response(2.0, tau=0.5), 4))
print("2% 调节时间 ≈", signal_utils.settling_time(0.5), "s")
print("模块文件:", signal_utils.__file__)

y(2s, tau=0.5) = 0.9817
2% 调节时间 ≈ 2.0 s
模块文件: /data/wangf/robot_rl_learn/runs/00_basic/b02/modules/signal_utils.py


## 小结与衔接

- 容器选择：数据流用 `list`、固定参数组用 `tuple`、参数表/配置用 `dict`、去重用 `set`；
- 推导式 + `enumerate`/`zip` 让数据处理代码缩短一半；
- 参数存 JSON、结果存 CSV、日志存 txt，产物一律进 `runs/`；
- `try/except` 给 IO 代码上保险；函数复用就收敛成 `.py` 模块。

**下周 B03**：告别 `for` 循环逐点计算——NumPy 向量化，一行代码算完一百万个采样点。

---

## ✏️ 练习

> 规则：先独立完成，再点开折叠的参考答案核对。

**练习 1（★，10 分钟，10 分）——报警样本筛选**
给定 `temps = [23.1, 79.8, 81.2, 25.0, 90.5, 60.0, 82.3]`，
(a) 用列表推导式取出所有 `> 80.0` 的报警样本；
(b) 用 `enumerate` 打印每个报警样本的**原始下标**和数值（提示：`if` 放推导式里或配合循环）。
**交付物**：打印下标与数值的 cell。

**练习 2（★，10 分钟，10 分）——PID 参数表**
用 `zip` 把 `["P", "PI", "PID"]` 与 `[(1.0,0,0), (1.0,0.5,0), (1.0,0.5,0.05)]`
组成嵌套 dict（键为控制器名，值为 `{"Kp":..,"Ki":..,"Kd":..}` 的 dict），
再用循环打印成对齐的三行表格。
**交付物**：嵌套 dict + 打印结果。

**练习 3（★★，20 分钟，20 分）——仿真数据落盘再读回**
(a) 用 B01 的欧拉法仿真一阶环节（$K=2,\tau=0.5,T_s=0.01$，5 s），
把 `(t, y)` 写入 `runs/00_basic/b02/ex3.csv`；
(b) 用 `try/except` 读回该文件，计算 $t \ge 4$ s 部分 $y$ 的平均值（应接近 2.0）；
(c) 故意读一个不存在的路径，验证兜底逻辑生效。
**交付物**：写文件 + 读回计算 + 异常兜底三个片段。

**练习 4（★，5 分钟，10 分）——概念辨析**
各举一例说明：什么时候用 `tuple` 而不是 `list`？什么时候 `dict` 比 `list` 更合适？
**交付物**：markdown 两句话。

---

<details>
<summary>参考答案（做完再点开）</summary>

**练习 1**：

```python
temps = [23.1, 79.8, 81.2, 25.0, 90.5, 60.0, 82.3]
alarms = [t for t in temps if t > 80.0]
print(alarms)
for k, t in enumerate(temps):
    if t > 80.0:
        print(f"下标 {k}: {t}")
```

**练习 2**：

```python
names = ["P", "PI", "PID"]
gains = [(1.0, 0, 0), (1.0, 0.5, 0), (1.0, 0.5, 0.05)]
table = {n: dict(zip(["Kp", "Ki", "Kd"], g)) for n, g in zip(names, gains)}
for n, g in table.items():
    print(f"{n:>4}: Kp={g['Kp']:.2f} Ki={g['Ki']:.2f} Kd={g['Kd']:.2f}")
```

**练习 3**：

```python
K, tau, Ts = 2.0, 0.5, 0.01
y, ys = 0.0, []
for k in range(501):
    ys.append((k * Ts, y))
    y += Ts * (K - y) / tau
path = DATA / "ex3.csv"
with path.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["t", "y"]); w.writerows(ys)

try:
    with path.open(encoding="utf-8") as f:
        rows = list(csv.reader(f))[1:]
    tail = [float(r[1]) for r in rows if float(r[0]) >= 4.0]
    print("t>=4s 平均值:", sum(tail) / len(tail))   # ≈ 2.0
except FileNotFoundError:
    print("文件不存在")
```

**练习 4**：`tuple` 用于「定义后不该改变」的固定组合（如电机参数 `(J, b, Kt)`、坐标 `(x, y)`），
不可变性防止误改；`dict` 用于「按名字查找」的场合（如 `plant["tau"]`），
比「`list[2]` 表示 tau」可读性好得多，也不怕顺序记错。

</details>

---

## 延伸阅读

- [Python 官方教程：第 5 章 数据结构](https://docs.python.org/zh-cn/3/tutorial/datastructures.html)
- [pathlib 文档](https://docs.python.org/zh-cn/3/library/pathlib.html)
- [csv 模块](https://docs.python.org/zh-cn/3/library/csv.html) / [json 模块](https://docs.python.org/zh-cn/3/library/json.html)
- [Python 异常处理官方教程](https://docs.python.org/zh-cn/3/tutorial/errors.html)